## Notebook 02 — Descriptive Exploration

Project: Ten-Year Haemoglobin Genotype Surveillance in a Nigerian University Cohort (Bowen University, 2015–2024, n = 8,890)

### Purpose

This notebook describes the cohort prior to any inferential analysis. It generates the descriptive statistics used in the Results section, including overall genotype and clinical-category distributions, sex composition, year-level breakdowns, and an age summary for the subset of records where age is available (2020–2024).

### Outputs

The notebook produces:

* Overall distributions of genotype, clinical category, and sex (counts and percentages)
* Cross-tabulations of sex with genotype and clinical category
* Yearly distributions of genotype, clinical category, and sex
* Age summary for 2020–2024 (subset with recorded age)

All tables are exported to `outputs/tables/` with the prefix `02_`.

Inferential analyses (chi-square tests and regression models) are performed in Notebooks 04 and 06. This notebook is limited to descriptive statistics.


## Cell 1 — Environment setup

In [8]:
# Cell 1: Environment setup
from pathlib import Path
import pandas as pd
import numpy as np

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed" / "processed_genotype_data.csv"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tables"
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)

GENO_ORDER = ["AA", "AC", "AS", "CC", "SC", "SS"]   # alphabetical, as in the source reports
CLIN_ORDER = ["Normal", "Carrier", "Disease"]

print("Environment configured.")

Environment configured.


## Cell 2 — Load the analytical dataset

In [9]:
# Cell 2: Load the validated dataset produced and checked in Notebook 01
df = pd.read_csv(DATA_PROCESSED, dtype={"ID": str, "Sex": str, "Genotype": str,
                                        "Clinical_Category": str})
df["Year"] = df["Year"].astype(int)
print(f"Loaded {len(df):,} records spanning {df['Year'].min()}-{df['Year'].max()}.")

Loaded 8,890 records spanning 2015-2024.


## Cell 3 — Overall distributions

In [10]:
# Cell 3: Overall genotype, clinical-category, and sex distributions

# Genotype
geno = (df["Genotype"].value_counts().reindex(GENO_ORDER).rename("Count").to_frame())
geno["Percent"] = (geno["Count"] / len(df) * 100).round(2)
print("Genotype distribution:")
display(geno)

# Clinical category
clin = (df["Clinical_Category"].value_counts().reindex(CLIN_ORDER).rename("Count").to_frame())
clin["Percent"] = (clin["Count"] / len(df) * 100).round(2)
print("Clinical-category distribution:")
display(clin)

# Sex
sex = (df["Sex"].value_counts().rename("Count").to_frame())
sex["Percent"] = (sex["Count"] / len(df) * 100).round(2)
print("Sex distribution:")
display(sex)

Genotype distribution:


,Count,Percent
Genotype,,
AA,6677,75.11
AC,292,3.28
AS,1814,20.40
CC,2,0.02
SC,42,0.47
SS,63,0.71


Clinical-category distribution:


,Count,Percent
Clinical_Category,,
Normal,6677,75.11
Carrier,2106,23.69
Disease,107,1.20


Sex distribution:


,Count,Percent
Sex,,
F,5118,57.57
M,3772,42.43


## Cell 4 — Distribution by sex

In [11]:
# Cell 4: Genotype and clinical category by sex

# Counts: Genotype (rows) x Sex (cols)
geno_by_sex = pd.crosstab(df["Genotype"], df["Sex"]).reindex(GENO_ORDER)
geno_by_sex = geno_by_sex[["F", "M"]]
print("Genotype counts by sex:")
display(geno_by_sex)

# Percentages within each sex (each column sums to 100)
geno_pct_by_sex = (geno_by_sex / geno_by_sex.sum() * 100).round(2)
print("Genotype percentages by sex (within-sex):")
display(geno_pct_by_sex)

# Clinical category by sex (percentages within sex)
clin_by_sex = pd.crosstab(df["Clinical_Category"], df["Sex"]).reindex(CLIN_ORDER)[["F", "M"]]
clin_pct_by_sex = (clin_by_sex / clin_by_sex.sum() * 100).round(2)
print("Clinical-category percentages by sex (within-sex):")
display(clin_pct_by_sex)

Genotype counts by sex:


Sex,F,M
Genotype,,
AA,3853,2824
AC,163,129
AS,1043,771
CC,1,1
SC,21,21
SS,37,26


Genotype percentages by sex (within-sex):


Sex,F,M
Genotype,,
AA,75.28,74.87
AC,3.18,3.42
AS,20.38,20.44
CC,0.02,0.03
SC,0.41,0.56
SS,0.72,0.69


Clinical-category percentages by sex (within-sex):


Sex,F,M
Clinical_Category,,
Normal,75.28,74.87
Carrier,23.56,23.86
Disease,1.15,1.27


## Cell 5 — Distribution by year (basis of Figure 1)

In [12]:
# Cell 5: Genotype and clinical category by year

# Genotype counts by year, with an 'All' total column and an 'All' total row
geno_by_year = pd.crosstab(df["Year"], df["Genotype"])[GENO_ORDER]
geno_by_year["All"] = geno_by_year.sum(axis=1)
geno_by_year.loc["All"] = geno_by_year.sum(axis=0)
print("Genotype counts by year (with totals):")
display(geno_by_year)

# Genotype percentages by year (row-wise; each year sums to ~100)
geno_pct_by_year = pd.crosstab(df["Year"], df["Genotype"], normalize="index")[GENO_ORDER] * 100
geno_pct_by_year = geno_pct_by_year.round(2)
print("Genotype percentages by year:")
display(geno_pct_by_year)

# Clinical category percentages by year
clin_pct_by_year = pd.crosstab(df["Year"], df["Clinical_Category"],
                               normalize="index")[["Carrier", "Disease", "Normal"]] * 100
clin_pct_by_year = clin_pct_by_year.round(2)
print("Clinical-category percentages by year:")
display(clin_pct_by_year)

Genotype counts by year (with totals):


Genotype,AA,AC,AS,CC,SC,SS,All
Year,,,,,,,
2015,757,40,209,1,5,8,1020
2016,176,9,62,0,1,2,250
2017,242,13,52,0,1,3,311
2018,662,34,212,0,7,8,923
2019,587,17,148,0,4,8,764
2020,563,30,152,0,2,2,749
2021,651,25,163,1,6,3,849
2022,1008,41,279,0,4,7,1339
2023,936,40,264,0,5,8,1253


Genotype percentages by year:


Genotype,AA,AC,AS,CC,SC,SS
Year,,,,,,
2015,74.22,3.92,20.49,0.10,0.49,0.78
2016,70.40,3.60,24.80,0.00,0.40,0.80
2017,77.81,4.18,16.72,0.00,0.32,0.96
2018,71.72,3.68,22.97,0.00,0.76,0.87
2019,76.83,2.23,19.37,0.00,0.52,1.05
2020,75.17,4.01,20.29,0.00,0.27,0.27
2021,76.68,2.94,19.20,0.12,0.71,0.35
2022,75.28,3.06,20.84,0.00,0.30,0.52
2023,74.70,3.19,21.07,0.00,0.40,0.64


Clinical-category percentages by year:


Clinical_Category,Carrier,Disease,Normal
Year,,,
2015,24.41,1.37,74.22
2016,28.40,1.20,70.40
2017,20.90,1.29,77.81
2018,26.65,1.63,71.72
2019,21.60,1.57,76.83
2020,24.30,0.53,75.17
2021,22.14,1.18,76.68
2022,23.90,0.82,75.28
2023,24.26,1.04,74.70


## Cell 6 — Sex distribution by year

In [13]:
# Cell 6: Sex percentages by year
sex_pct_by_year = pd.crosstab(df["Year"], df["Sex"], normalize="index") * 100
sex_pct_by_year = sex_pct_by_year.rename(columns={"M": "Male", "F": "Female"})
sex_pct_by_year = sex_pct_by_year[["Male", "Female"]].round(2)
print("Sex percentages by year:")
display(sex_pct_by_year)

Sex percentages by year:


Sex,Male,Female
Year,,
2015,38.92,61.08
2016,33.60,66.40
2017,58.52,41.48
2018,42.36,57.64
2019,38.48,61.52
2020,39.92,60.08
2021,35.69,64.31
2022,44.14,55.86
2023,44.21,55.79


## Cell 7 — Age summary (subset: 2020–2024)

Age was recorded only from 2020 onward; the 2015–2019 exports contain no age field.
The summary below is therefore based on the **subset** of the cohort with a recorded age
(years 2020–2024), not on all 8,890 students. One 2022 record carried an out-of-range
value (1116) and is excluded by restricting to the plausible range reported in the
manuscript (14–42). This is stated explicitly so the statistic is not mistaken for a
whole-cohort figure.

In [14]:
# Cell 7: Age summary from the raw exports that recorded age (2020-2024)

# Each yearly export used a different layout; (age column index, header row index):
AGE_SPECS = {
    2020: (3, 1),
    2021: (3, 1),
    2022: (2, 0),
    2023: (3, 1),
    2024: (1, 1),
}

ages = []
for year, (col, header_row) in AGE_SPECS.items():
    raw = pd.read_csv(DATA_RAW / f"Genotype_{year}.csv", header=None,
                      dtype=str, keep_default_na=False)
    for v in raw.iloc[header_row + 1:, col].tolist():
        v = str(v).strip()
        if v.replace(".", "", 1).isdigit():
            ages.append(float(v))

ages = np.array(ages)
plausible = ages[(ages >= 14) & (ages <= 42)]   # manuscript range; removes the 1116 typo

age_summary = pd.DataFrame({
    "Statistic": ["n (with recorded age)", "Mean", "SD", "Min", "Max"],
    "Value": [len(plausible), round(plausible.mean(), 2),
              round(plausible.std(ddof=1), 2), int(plausible.min()), int(plausible.max())],
})
print("Age summary (2020-2024 subset, ages 14-42):")
display(age_summary)
print(f"Subset covers {len(plausible):,} of {len(df):,} students "
      f"({len(plausible)/len(df)*100:.1f}% of the cohort).")

Age summary (2020-2024 subset, ages 14-42):


,Statistic,Value
0,n (with recorded age),5812.00
1,Mean,16.84
2,SD,1.67
3,Min,14.00
4,Max,42.00


Subset covers 5,812 of 8,890 students (65.4% of the cohort).


## Cell 8 — Export tables

In [15]:
# Cell 8: Save every table to outputs/tables/ with an 02_ prefix

geno.to_csv(OUTPUTS_TABLES / "02_genotype_distribution_overall.csv")
clin.to_csv(OUTPUTS_TABLES / "02_clinical_category_distribution_overall.csv")
sex.to_csv(OUTPUTS_TABLES / "02_sex_distribution_overall.csv")
geno_by_sex.to_csv(OUTPUTS_TABLES / "02_genotype_counts_by_sex.csv")
geno_pct_by_sex.to_csv(OUTPUTS_TABLES / "02_genotype_percentages_by_sex.csv")
clin_pct_by_sex.to_csv(OUTPUTS_TABLES / "02_clinical_category_percentages_by_sex.csv")
geno_by_year.to_csv(OUTPUTS_TABLES / "02_genotype_counts_by_year_with_totals.csv")
geno_pct_by_year.to_csv(OUTPUTS_TABLES / "02_genotype_percentages_by_year.csv")
clin_pct_by_year.to_csv(OUTPUTS_TABLES / "02_clinical_category_percentages_by_year.csv")
sex_pct_by_year.to_csv(OUTPUTS_TABLES / "02_sex_percentages_by_year.csv")
age_summary.to_csv(OUTPUTS_TABLES / "02_age_summary_subset.csv", index=False)

saved = sorted(p.name for p in OUTPUTS_TABLES.glob("02_*.csv"))
print(f"Saved {len(saved)} tables:")
for s in saved:
    print(" ", s)

Saved 11 tables:
  02_age_summary_subset.csv
  02_clinical_category_distribution_overall.csv
  02_clinical_category_percentages_by_sex.csv
  02_clinical_category_percentages_by_year.csv
  02_genotype_counts_by_sex.csv
  02_genotype_counts_by_year_with_totals.csv
  02_genotype_distribution_overall.csv
  02_genotype_percentages_by_sex.csv
  02_genotype_percentages_by_year.csv
  02_sex_distribution_overall.csv
  02_sex_percentages_by_year.csv


## Summary

The cohort is described in full: overall composition, the sex and year cross-tabulations
that feed Figure 1, and an age summary that is explicitly scoped to the years recording
age. Every table is exported for downstream use and for the supplementary materials.

**Next:** Notebook 03 produces the publication figures 